# 8-Tier Emissions Sankey Chart (Global-Ranked)

**Ranking logic:** Sectors are ranked *globally* by total emissions at each tier.
- Tier 1: top 9 globally → rest into Other
- Tier 2: top 5 globally (summed across all T1 parents) → rest into Other
- Tier 3: top 5 globally (summed across all T2 parents) → rest into Other
- Other chains: T1 Other → T2 Other → T3 Other

**Set `FILE_PATH` and `SECTOR_NAME` before running.**


In [9]:
# ─── CONFIGURATION ─────────────────────────────────────────────────────────
FILE_PATH   = "TASA-EFX_rLCA_Abrasive-Material_KOR_2023_v1.0_DEMO.xlsx"
SECTOR_NAME = "Abrasive Material"
YEAR        = "2023"

TOP_N_T1 = 10   # Top N Tier-1 sectors shown globally
TOP_N_T2 = 5   # Top N Tier-2 sectors shown globally (summed across all T1 parents)
TOP_N_T3 = 5   # Top N Tier-3 sectors shown globally (summed across all T2 parents)
# ───────────────────────────────────────────────────────────────────────────


In [10]:
import json
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# ── 1. Load top-level totals from Sector Emissions Factors ───────────────────
ef = pd.read_excel(FILE_PATH, sheet_name="Sector Emissions Factors",
                   header=0, dtype=str)
ef.columns = [str(c).strip() for c in ef.columns]

# Filter to the chosen sector row
sector_row = ef[ef.iloc[:, 0].str.strip() == SECTOR_NAME].iloc[0]

# B2 = col index 1 (0-based), C2=2, D2=3, E2=4
total_ei = float(sector_row.iloc[1])
s1_total = float(sector_row.iloc[2])
s2_total = float(sector_row.iloc[3])
s3_total = float(sector_row.iloc[4])

print(f"Sector : {SECTOR_NAME}")
print(f"Total  : {total_ei:.4f}")
print(f"Scope 1: {s1_total:.4f}")
print(f"Scope 2: {s2_total:.4f}")
print(f"Scope 3: {s3_total:.4f}")

Sector : Abrasive Material
Total  : 460.2669
Scope 1: 41.6536
Scope 2: 55.6447
Scope 3: 362.9686


In [11]:
# ── 2. Load Tier 1 Pathway Analysis ─────────────────────────────────────────
t1_raw = pd.read_excel(FILE_PATH, sheet_name="Tier_1 Pathway Analysis",
                       header=0, dtype=str)
t1_raw.columns = [str(c).strip() for c in t1_raw.columns]

# Filter to sector
sector_col = t1_raw.columns[0]        # "Sector"
t1_sec     = t1_raw[t1_raw[sector_col].str.strip() == SECTOR_NAME].copy()

# Identify column names (flexible – finds by keyword)
def find_col(df, *keywords):
    """Return first column whose name contains all keywords (case-insensitive)."""
    kws = [k.lower() for k in keywords]
    for c in df.columns:
        cl = c.lower()
        if all(k in cl for k in kws):
            return c
    raise KeyError(f"No column matching {keywords} in {list(df.columns)}")

T1_SECTOR_COL = find_col(t1_sec, "tier 1", "input")
T1_TOTAL_COL  = find_col(t1_sec, "tier 1", "total")
T1_S1_COL     = find_col(t1_sec, "tier 1", "scope 1")
T1_S2_COL     = find_col(t1_sec, "tier 1", "scope 2")
T1_S3_COL     = find_col(t1_sec, "tier 1", "scope 3")

for c in [T1_TOTAL_COL, T1_S1_COL, T1_S2_COL, T1_S3_COL]:
    t1_sec[c] = pd.to_numeric(t1_sec[c], errors="coerce")

# De-dup on Tier 1 Input Sector, keep max total if duplicates exist
t1_sec = (t1_sec
          .dropna(subset=[T1_TOTAL_COL])
          .sort_values(T1_TOTAL_COL, ascending=False)
          .drop_duplicates(subset=[T1_SECTOR_COL]))

# Split Other vs non-Other, take top N non-Other, then bucket the rest
is_other_mask = t1_sec[T1_SECTOR_COL].str.strip().str.lower() == "other"
t1_non_other  = t1_sec[~is_other_mask].head(TOP_N_T1)
t1_rest       = t1_sec[~is_other_mask].iloc[TOP_N_T1:]   # sectors beyond top-N
t1_explicit_other = t1_sec[is_other_mask]

# Combine rest + explicit Other rows into a single "Other" bucket
all_other = pd.concat([t1_rest, t1_explicit_other])
if not all_other.empty:
    other_row = pd.DataFrame([{
        T1_SECTOR_COL: "Other",
        T1_TOTAL_COL:  all_other[T1_TOTAL_COL].sum(),
        T1_S1_COL:     all_other[T1_S1_COL].sum(),
        T1_S2_COL:     all_other[T1_S2_COL].sum(),
        T1_S3_COL:     all_other[T1_S3_COL].sum(),
    }])
    t1_final = pd.concat([t1_non_other, other_row]).reset_index(drop=True)
else:
    t1_final = t1_non_other.reset_index(drop=True)

print(f"Tier 1 sectors shown: {len(t1_final)}")
print(f"Tier 1 sectors sum: {t1_final[T1_TOTAL_COL].sum()}")
t1_final[[T1_SECTOR_COL, T1_TOTAL_COL]].head(12)

Tier 1 sectors shown: 11
Tier 1 sectors sum: 362.9686425235485


,Tier 1 Input Sector,Tier 1 Total Emissions Intensity
0,Basic Inorganic Compounds,38.401632
1,Other Fiber Fabrics,36.241310
2,Adhesive and Gelatin,24.677415
3,Synthetic Resin,24.205417
4,Manufacturing Equipment Repair,23.472489
5,Other Nonmetallic Mineral Products,22.073310
6,Natural and Chemical Fiber Fabrics,21.431768
7,Abrasive Material,19.253848
8,Road Haulage Transportation,17.347114
9,Other Chemical Products,16.517463


In [12]:
# ── 3. Load Tier 2 Pathway Analysis (global ranking) ────────────────────────
t2_raw = pd.read_excel(FILE_PATH, sheet_name="Tier_2 Pathway Analysis",
                       header=0, dtype=str)
t2_raw.columns = [str(c).strip() for c in t2_raw.columns]

t2_sec = t2_raw[t2_raw.iloc[:, 0].str.strip() == SECTOR_NAME].copy()

T2_T1_COL     = find_col(t2_sec, "tier 1", "input")
T2_SECTOR_COL = find_col(t2_sec, "tier 2", "input")
T2_TOTAL_COL  = find_col(t2_sec, "tier 2", "total")
T2_S1_COL     = find_col(t2_sec, "tier 2", "scope 1")
T2_S2_COL     = find_col(t2_sec, "tier 2", "scope 2")
T2_S3_COL     = find_col(t2_sec, "tier 2", "scope 3")

for c in [T2_TOTAL_COL, T2_S1_COL, T2_S2_COL, T2_S3_COL]:
    t2_sec[c] = pd.to_numeric(t2_sec[c], errors="coerce")
t2_sec = t2_sec.dropna(subset=[T2_TOTAL_COL])

# ── Global ranking: sum each T2 sector name across ALL T1 parents ─────────────
t2_is_other = t2_sec[T2_SECTOR_COL].str.strip().str.lower() == "other"
t2_named    = t2_sec[~t2_is_other].copy()

# Sum total per T2 sector name globally
t2_global_totals = (
    t2_named.groupby(t2_named[T2_SECTOR_COL].str.strip())[T2_TOTAL_COL]
    .sum().sort_values(ascending=False)
)
top_t2_names = set(t2_global_totals.head(TOP_N_T2).index)
print(f"Global top-{TOP_N_T2} Tier-2 sectors: {sorted(top_t2_names)}")

# ── Build t2_by_t1: group ALL T2 sheet rows by T1 parent ─────────────────────
# Any T1 parent not in t1_final is remapped to "Other" — no silent exclusions.
# t2_by_t1[t1_name] = list of {label, total, s1, s2, s3}

t1_final_names = set(t1_final[T1_SECTOR_COL].str.strip())
t2_by_t1 = {}

for t1_name, subset in t2_sec.groupby(t2_sec[T2_T1_COL].str.strip()):
    # T1 parents not in t1_final fold into the "Other" bucket
    bucket = t1_name if t1_name in t1_final_names else "Other"

    subset = subset.drop_duplicates(subset=[T2_SECTOR_COL])
    rows = []
    other_total = other_s1 = other_s2 = other_s3 = 0.0

    for _, r in subset.iterrows():
        lbl = str(r[T2_SECTOR_COL]).strip()
        tot = float(r[T2_TOTAL_COL])
        s1  = float(r[T2_S1_COL])
        s2  = float(r[T2_S2_COL])
        s3  = float(r[T2_S3_COL])

        if lbl.lower() == "other" or lbl not in top_t2_names:
            other_total += tot; other_s1 += s1; other_s2 += s2; other_s3 += s3
        else:
            rows.append({"label": lbl, "total": tot, "s1": s1, "s2": s2, "s3": s3})

    if other_total > 0:
        rows.append({"label": "Other", "total": other_total,
                     "s1": other_s1, "s2": other_s2, "s3": other_s3})

    if rows:
        if bucket not in t2_by_t1:
            t2_by_t1[bucket] = rows
        else:
            t2_by_t1[bucket].extend(rows)

print(f"Tier 2 groups (by T1 parent): {len(t2_by_t1)}")
print(f"Tier 2 total sum: {sum(r['total'] for rows in t2_by_t1.values() for r in rows):.4f}")
print("T2 groups built:", {k: [r["label"] for r in v] for k, v in list(t2_by_t1.items())[:3]})

t2_flat = pd.DataFrame([
    {"T1 Parent": t1, "T2 Sector": r["label"], "Total": r["total"],
     "S1": r["s1"], "S2": r["s2"], "S3": r["s3"]}
    for t1, rows in t2_by_t1.items() for r in rows
])
t2_flat.head(12)

Global top-5 Tier-2 sectors: ['Basic Inorganic Compounds', 'Manufacturing Equipment Repair', 'Petrochemical Intermediate Products', 'Road Haulage Transportation', 'Synthetic Resin']
Tier 2 groups (by T1 parent): 11
Tier 2 total sum: 250.7685
T2 groups built: {'Abrasive Material': ['Basic Inorganic Compounds', 'Synthetic Resin', 'Manufacturing Equipment Repair', 'Road Haulage Transportation', 'Petrochemical Intermediate Products', 'Other'], 'Adhesive and Gelatin': ['Synthetic Resin', 'Petrochemical Intermediate Products', 'Basic Inorganic Compounds', 'Road Haulage Transportation', 'Manufacturing Equipment Repair', 'Other'], 'Other': ['Road Haulage Transportation', 'Other', 'Other', 'Road Haulage Transportation', 'Manufacturing Equipment Repair', 'Synthetic Resin', 'Other', 'Other', 'Manufacturing Equipment Repair', 'Road Haulage Transportation', 'Synthetic Resin', 'Other', 'Road Haulage Transportation', 'Other', 'Road Haulage Transportation', 'Basic Inorganic Compounds', 'Other', 'Road 

,T1 Parent,T2 Sector,Total,S1,S2,S3
0,Abrasive Material,Basic Inorganic Compounds,1.606414,0.236881,0.488178,0.881355
1,Abrasive Material,Synthetic Resin,1.012559,0.198751,0.122661,0.691147
2,Abrasive Material,Manufacturing Equipment Repair,0.981899,0.087732,0.149710,0.744457
3,Abrasive Material,Road Haulage Transportation,0.725663,0.460520,0.004166,0.260978
4,Abrasive Material,Petrochemical Intermediate Products,0.172718,0.029905,0.014097,0.128715
5,Abrasive Material,Other,9.389905,1.190322,1.352270,6.847312
6,Adhesive and Gelatin,Synthetic Resin,3.419530,0.671206,0.414240,2.334084
7,Adhesive and Gelatin,Petrochemical Intermediate Products,2.656312,0.459929,0.216809,1.979574
8,Adhesive and Gelatin,Basic Inorganic Compounds,0.719360,0.106076,0.218609,0.394675
9,Adhesive and Gelatin,Road Haulage Transportation,0.305853,0.194100,0.001756,0.109997


In [13]:
# ── 4. Load Tier 3 Pathway Analysis (global ranking) ────────────────────────
t3_raw = pd.read_excel(FILE_PATH, sheet_name="Tier_3 Pathway Analysis",
                       header=0, dtype=str)
t3_raw.columns = [str(c).strip() for c in t3_raw.columns]

t3_sec = t3_raw[t3_raw.iloc[:, 0].str.strip() == SECTOR_NAME].copy()

T3_T1_COL     = find_col(t3_sec, "tier 1", "input")
T3_T2_COL     = find_col(t3_sec, "tier 2", "input")
T3_SECTOR_COL = find_col(t3_sec, "tier 3", "input")
T3_TOTAL_COL  = find_col(t3_sec, "tier 3", "total")
T3_S1_COL     = find_col(t3_sec, "tier 3", "scope 1")
T3_S2_COL     = find_col(t3_sec, "tier 3", "scope 2")
T3_S3_COL     = find_col(t3_sec, "tier 3", "scope 3")

for c in [T3_TOTAL_COL, T3_S1_COL, T3_S2_COL, T3_S3_COL]:
    t3_sec[c] = pd.to_numeric(t3_sec[c], errors="coerce")
t3_sec = t3_sec.dropna(subset=[T3_TOTAL_COL])

# ── Global ranking: sum each T3 sector name across ALL (T1, T2) parents ───────
t3_is_other = t3_sec[T3_SECTOR_COL].str.strip().str.lower() == "other"
t3_named    = t3_sec[~t3_is_other].copy()

t3_global_totals = (
    t3_named.groupby(t3_named[T3_SECTOR_COL].str.strip())[T3_TOTAL_COL]
    .sum().sort_values(ascending=False)
)
top_t3_names = set(t3_global_totals.head(TOP_N_T3).index)
print(f"Global top-{TOP_N_T3} Tier-3 sectors: {sorted(top_t3_names)}")

# ── Build t3_by_t2: keyed by (t1_name, t2_name) ───────────────────────────────
# For each (T1, T2) pair, keep rows whose T3 label is in top_t3_names; rest → Other
t3_by_t2 = {}

for t1_name, t2_rows in t2_by_t1.items():
    for t2_row in t2_rows:
        t2_name = t2_row["label"]
        key = (t1_name, t2_name)

        # Other chains: T2-Other → T3-Other
        # Pass ALL scope values so the chain stays balanced.
        if t2_name == "Other" or t1_name == "Other":
            t3_by_t2[key] = [{
                "label": "Other",
                "total": t2_row["total"],
                "s1": t2_row["s1"],
                "s2": t2_row["s2"],
                "s3": t2_row["s3"],
            }]
            continue

        subset = t3_sec[
            (t3_sec[T3_T1_COL].str.strip() == t1_name) &
            (t3_sec[T3_T2_COL].str.strip() == t2_name)
        ].copy().drop_duplicates(subset=[T3_SECTOR_COL])

        rows = []
        other_total = other_s1 = other_s2 = other_s3 = 0.0

        for _, r in subset.iterrows():
            lbl = str(r[T3_SECTOR_COL]).strip()
            tot = float(r[T3_TOTAL_COL])
            s1  = float(r[T3_S1_COL])
            s2  = float(r[T3_S2_COL])
            s3  = float(r[T3_S3_COL])

            if lbl.lower() == "other" or lbl not in top_t3_names:
                other_total += tot; other_s1 += s1; other_s2 += s2; other_s3 += s3
            else:
                rows.append({"label": lbl, "total": tot, "s1": s1, "s2": s2, "s3": s3})

        if other_total > 0:
            rows.append({"label": "Other", "total": other_total,
                         "s1": other_s1, "s2": other_s2, "s3": other_s3})

        if rows:
            t3_by_t2[key] = rows

print(f"Tier-3 groups loaded: {len(t3_by_t2)}")
for k, v in list(t3_by_t2.items())[:4]:
    print(f"  {k}: {[r['label'] for r in v]}")


Global top-5 Tier-3 sectors: ['Aliphatic Base Oil', 'Aromatic Base Oil', 'Crude Oil', 'Marketing Research and Management Support Service', 'Petrochemical Intermediate Products']
Tier-3 groups loaded: 56
  ('Abrasive Material', 'Basic Inorganic Compounds'): ['Aromatic Base Oil', 'Marketing Research and Management Support Service', 'Aliphatic Base Oil', 'Petrochemical Intermediate Products', 'Other']
  ('Abrasive Material', 'Synthetic Resin'): ['Aliphatic Base Oil', 'Petrochemical Intermediate Products', 'Marketing Research and Management Support Service', 'Aromatic Base Oil', 'Crude Oil', 'Other']
  ('Abrasive Material', 'Manufacturing Equipment Repair'): ['Marketing Research and Management Support Service', 'Other']
  ('Abrasive Material', 'Road Haulage Transportation'): ['Marketing Research and Management Support Service', 'Other']


In [14]:
# ── 5. Assemble JSON payload ─────────────────────────────────────────────────

def pct(val, base):
    return round(val / base * 100, 2) if base else 0.0

scopes = [
    {"label": "Scope 1", "val": round(s1_total, 4), "pct": pct(s1_total, total_ei), "color": "#3A86C8"},
    {"label": "Scope 2", "val": round(s2_total, 4), "pct": pct(s2_total, total_ei), "color": "#2BAE96"},
    {"label": "Scope 3", "val": round(s3_total, 4), "pct": pct(s3_total, total_ei), "color": "#1B4F8A"},
]

tier1_data = []
for _, row in t1_final.iterrows():
    t_total=float(row[T1_TOTAL_COL]); t_s1=float(row[T1_S1_COL])
    t_s2=float(row[T1_S2_COL]); t_s3=float(row[T1_S3_COL])
    tier1_data.append({
        "label": str(row[T1_SECTOR_COL]).strip(), "val": round(t_total,4),
        "pct": pct(t_total,total_ei),
        "s1_val": round(t_s1,4), "s1": pct(t_s1,total_ei),
        "s2_val": round(t_s2,4), "s2": pct(t_s2,total_ei),
        "s3_val": round(t_s3,4), "s3": pct(t_s3,total_ei),
    })

# tier2_data: keyed by t1_label; each entry is a list of sector rows
# Each sector row now includes "label" which may appear under multiple T1 parents
tier2_data = {}
for t1_label, rows in t2_by_t1.items():
    tier2_data[t1_label] = [{
        "label": r["label"], "val": round(r["total"],4), "pct": pct(r["total"],total_ei),
        "s1_val": round(r["s1"],4), "s1": pct(r["s1"],total_ei),
        "s2_val": round(r["s2"],4), "s2": pct(r["s2"],total_ei),
        "s3_val": round(r["s3"],4), "s3": pct(r["s3"],total_ei),
    } for r in rows]

# tier3_data: keyed by "t1|||t2"
tier3_data = {}
for (t1_label, t2_label), rows in t3_by_t2.items():
    key = t1_label + "|||" + t2_label
    tier3_data[key] = [{
        "label": r["label"], "val": round(r["total"],4), "pct": pct(r["total"],total_ei),
        "s1_val": round(r["s1"],4), "s1": pct(r["s1"],total_ei),
        "s2_val": round(r["s2"],4), "s2": pct(r["s2"],total_ei),
        "s3_val": round(r["s3"],4), "s3": pct(r["s3"],total_ei),
    } for r in rows]

payload = json.dumps({
    "sector": SECTOR_NAME, "total_ei": round(total_ei,4),
    "scopes": scopes, "tier1": tier1_data,
    "tier2": tier2_data, "tier3": tier3_data,
})

print("Payload assembled:")
print(f"  Tier1 sectors: {len(tier1_data)}")
print(f"  Tier2 groups (by T1): {len(tier2_data)}, total rows: {sum(len(v) for v in tier2_data.values())}")
print(f"  Tier3 groups (by T1|T2): {len(tier3_data)}, total rows: {sum(len(v) for v in tier3_data.values())}")
# Show unique T2 sector names to confirm global top-N is working
t2_labels = sorted(set(r["label"] for rows in tier2_data.values() for r in rows))
t3_labels = sorted(set(r["label"] for rows in tier3_data.values() for r in rows))
print(f"  Unique T2 labels: {t2_labels}")
print(f"  Unique T3 labels: {t3_labels}")


Payload assembled:
  Tier1 sectors: 11
  Tier2 groups (by T1): 11, total rows: 155
  Tier3 groups (by T1|T2): 56, total rows: 174
  Unique T2 labels: ['Basic Inorganic Compounds', 'Manufacturing Equipment Repair', 'Other', 'Petrochemical Intermediate Products', 'Road Haulage Transportation', 'Synthetic Resin']
  Unique T3 labels: ['Aliphatic Base Oil', 'Aromatic Base Oil', 'Crude Oil', 'Marketing Research and Management Support Service', 'Other', 'Petrochemical Intermediate Products']


In [15]:
# ── SANKEY BALANCE CHECKS ──────────────────────────────────────────────────
# Verifies that Scope 1 + Scope 2 + Scope 3 sums match reported totals
# at each tier. Discrepancies here explain why the Sankey does not balance.

import textwrap

SEP  = "-" * 72
SEP2 = "=" * 72

def check_scope_sum(label, total_val, s1, s2, s3, tol=1e-6):
    """Print a row; flag if S1+S2+S3 != total."""
    scope_sum = s1 + s2 + s3
    diff      = scope_sum - total_val
    flag = "  ✓" if abs(diff) < tol else f"  ✗  DIFF = {diff:+.6f}"
    print(f"  {label:<40s}  S1={s1:.4f}  S2={s2:.4f}  S3={s3:.4f}")
    print(f"  {'':<40s}  Sum={scope_sum:.4f}  Total={total_val:.4f}{flag}")
    print()

# ── Baseline (Sector Emissions Factors) ─────────────────────────────────────
print(SEP2)
print("BASELINE (Sector Emissions Factors)")
print(SEP2)
check_scope_sum(SECTOR_NAME, total_ei, s1_total, s2_total, s3_total)

# ── Tier 1 ──────────────────────────────────────────────────────────────────
print(SEP2)
print("TIER 1 – per sector")
print(SEP2)
t1_s1_sum = t1_s2_sum = t1_s3_sum = t1_total_sum = 0.0
for _, row in t1_final.iterrows():
    t = str(row[T1_SECTOR_COL]).strip()
    tot = float(row[T1_TOTAL_COL])
    s1  = float(row[T1_S1_COL])
    s2  = float(row[T1_S2_COL])
    s3  = float(row[T1_S3_COL])
    check_scope_sum(t, tot, s1, s2, s3)
    t1_s1_sum += s1; t1_s2_sum += s2; t1_s3_sum += s3
    t1_total_sum += tot
print(SEP)
print("TIER 1 GRAND TOTAL")
check_scope_sum("All T1 sectors", t1_total_sum, t1_s1_sum, t1_s2_sum, t1_s3_sum)
print("  Compare against baseline:")
check_scope_sum("  Baseline vs T1 total", total_ei, t1_s1_sum, t1_s2_sum, t1_s3_sum,
                tol=abs(total_ei)*0.001)

# ── Tier 2 ──────────────────────────────────────────────────────────────────
print(SEP2)
print("TIER 2 – per T1 parent (summed across T2 children)")
print(SEP2)
t2_s1_sum = t2_s2_sum = t2_s3_sum = t2_total_sum = 0.0
for t1_name, t2_rows in t2_by_t1.items():
    g_s1 = sum(r["s1"] for r in t2_rows)
    g_s2 = sum(r["s2"] for r in t2_rows)
    g_s3 = sum(r["s3"] for r in t2_rows)
    g_tot = sum(r["total"] for r in t2_rows)
    # Compare against the T1 row it came from
    t1_row = t1_final[t1_final[T1_SECTOR_COL].str.strip() == t1_name]
    if len(t1_row):
        t1_ref_tot = float(t1_row.iloc[0][T1_TOTAL_COL])
        t1_ref_s1  = float(t1_row.iloc[0][T1_S1_COL])
        t1_ref_s2  = float(t1_row.iloc[0][T1_S2_COL])
        t1_ref_s3  = float(t1_row.iloc[0][T1_S3_COL])
        diff_tot = g_tot - t1_ref_tot
        diff_s3  = g_s3  - t1_ref_s3
        flag = "  ✓" if abs(diff_tot) < abs(t1_ref_tot)*0.001 + 1e-9 else f"  ✗  tot diff={diff_tot:+.4f}"
        s3flag = "  ✓" if abs(diff_s3) < abs(t1_ref_s3)*0.001 + 1e-9 else f"  ✗  S3 diff={diff_s3:+.4f}"
        print(f"  T1 parent: {t1_name}")
        print(f"    T2 children sum  tot={g_tot:.4f}  s1={g_s1:.4f}  s2={g_s2:.4f}  s3={g_s3:.4f}")
        print(f"    T1 row           tot={t1_ref_tot:.4f}  s1={t1_ref_s1:.4f}  s2={t1_ref_s2:.4f}  s3={t1_ref_s3:.4f}")
        print(f"    Total match:{flag}  S3 match:{s3flag}")
        print()
    t2_s1_sum += g_s1; t2_s2_sum += g_s2; t2_s3_sum += g_s3; t2_total_sum += g_tot
print(SEP)
print("TIER 2 GRAND TOTAL")
check_scope_sum("All T2 sectors", t2_total_sum, t2_s1_sum, t2_s2_sum, t2_s3_sum)

# ── Tier 3 ──────────────────────────────────────────────────────────────────
print(SEP2)
print("TIER 3 – per (T1, T2) parent (summed across T3 children)")
print(SEP2)
t3_s1_sum = t3_s2_sum = t3_s3_sum = t3_total_sum = 0.0
for (t1_name, t2_name), t3_rows in t3_by_t2.items():
    g_s1 = sum(r["s1"] for r in t3_rows)
    g_s2 = sum(r["s2"] for r in t3_rows)
    g_s3 = sum(r["s3"] for r in t3_rows)
    g_tot = sum(r["total"] for r in t3_rows)
    # Compare against T2 parent
    t2_parent = [r for r in t2_by_t1.get(t1_name, []) if r["label"] == t2_name]
    if t2_parent:
        p = t2_parent[0]
        diff_tot = g_tot - p["total"]
        diff_s3  = g_s3  - p["s3"]
        flag = "  ✓" if abs(diff_tot) < abs(p["total"])*0.001 + 1e-9 else f"  ✗  tot diff={diff_tot:+.4f}"
        s3flag = "  ✓" if abs(diff_s3) < abs(p["s3"])*0.001 + 1e-9 else f"  ✗  S3 diff={diff_s3:+.4f}"
        print(f"  ({t1_name} → {t2_name})")
        print(f"    T3 sum  tot={g_tot:.4f}  s1={g_s1:.4f}  s2={g_s2:.4f}  s3={g_s3:.4f}")
        print(f"    T2 ref  tot={p['total']:.4f}  s1={p['s1']:.4f}  s2={p['s2']:.4f}  s3={p['s3']:.4f}")
        print(f"    Total match:{flag}  S3 match:{s3flag}")
        print()
    t3_s1_sum += g_s1; t3_s2_sum += g_s2; t3_s3_sum += g_s3; t3_total_sum += g_tot
print(SEP)
print("TIER 3 GRAND TOTAL")
check_scope_sum("All T3 sectors", t3_total_sum, t3_s1_sum, t3_s2_sum, t3_s3_sum)

# ── Cross-tier summary ───────────────────────────────────────────────────────
print(SEP2)
print("CROSS-TIER SUMMARY  (S1+S2+S3 at each tier vs baseline)")
print(SEP2)
rows = [
    ("Baseline",   s1_total,  s2_total,  s3_total,  total_ei),
    ("Tier 1 sum", t1_s1_sum, t1_s2_sum, t1_s3_sum, t1_total_sum),
    ("Tier 2 sum", t2_s1_sum, t2_s2_sum, t2_s3_sum, t2_total_sum),
    ("Tier 3 sum", t3_s1_sum, t3_s2_sum, t3_s3_sum, t3_total_sum),
]
print(f"  {'Level':<12}  {'S1':>10}  {'S2':>10}  {'S3':>10}  {'Total':>10}  {'S1+S2+S3':>12}  Match?")
print("  " + "-"*80)
for label, s1, s2, s3, tot in rows:
    ssum = s1 + s2 + s3
    ok = abs(ssum - tot) < abs(tot)*0.001 + 1e-9
    mark = "✓" if ok else "✗"
    print(f"  {label:<12}  {s1:>10.4f}  {s2:>10.4f}  {s3:>10.4f}  {tot:>10.4f}  {ssum:>12.4f}  {mark}")


BASELINE (Sector Emissions Factors)
  Abrasive Material                         S1=41.6536  S2=55.6447  S3=362.9686
                                            Sum=460.2669  Total=460.2669  ✓

TIER 1 – per sector
  Basic Inorganic Compounds                 S1=5.6627  S2=11.6700  S3=21.0690
                                            Sum=38.4016  Total=38.4016  ✓

  Other Fiber Fabrics                       S1=3.4307  S2=5.7573  S3=27.0534
                                            Sum=36.2413  Total=36.2413  ✓

  Adhesive and Gelatin                      S1=1.6967  S2=1.4485  S3=21.5323
                                            Sum=24.6774  Total=24.6774  ✓

  Synthetic Resin                           S1=4.7512  S2=2.9322  S3=16.5220
                                            Sum=24.2054  Total=24.2054  ✓

  Manufacturing Equipment Repair            S1=2.0972  S2=3.5789  S3=17.7964
                                            Sum=23.4725  Total=23.4725  ✓

  Other Nonmetallic Minera

In [16]:
# ── 6. Render the 8-tier Sankey ──────────────────────────────────────────────

uid = "snk8t"

html = f"""
<div id="wrap-{uid}" style="font-family:'Segoe UI',Arial,sans-serif;">
  <div style="margin-bottom:10px;display:flex;gap:8px;align-items:center;flex-wrap:wrap;">
    <span style="font-size:11px;color:#4A6A8A;font-weight:600;">Scope view:</span>
    <button id="btn-detail-{uid}" onclick="setMode_{uid}('detail')"
      style="padding:5px 14px;border-radius:20px;border:none;cursor:pointer;font-size:11px;
             font-weight:600;background:#1B4F8A;color:#fff;">Broken Out</button>
    <button id="btn-summed-{uid}" onclick="setMode_{uid}('summed')"
      style="padding:5px 14px;border-radius:20px;border:2px solid #1B4F8A;cursor:pointer;font-size:11px;
             font-weight:600;background:#fff;color:#1B4F8A;">Summed</button>
    <span style="margin-left:12px;font-size:11px;color:#4A6A8A;font-weight:600;">Tier 2:</span>
    <button id="btn-t2combine-{uid}" onclick="setT2Combine_{uid}(true)"
      style="padding:5px 14px;border-radius:20px;border:none;cursor:pointer;font-size:11px;
             font-weight:600;background:#1B4F8A;color:#fff;">Combined</button>
    <button id="btn-t2split-{uid}" onclick="setT2Combine_{uid}(false)"
      style="padding:5px 14px;border-radius:20px;border:2px solid #1B4F8A;cursor:pointer;font-size:11px;
             font-weight:600;background:#fff;color:#1B4F8A;">Split</button>
    <span style="margin-left:12px;font-size:11px;color:#4A6A8A;font-weight:600;">Tier 3:</span>
    <button id="btn-t3combine-{uid}" onclick="setT3Combine_{uid}(true)"
      style="padding:5px 14px;border-radius:20px;border:none;cursor:pointer;font-size:11px;
             font-weight:600;background:#1B4F8A;color:#fff;">Combined</button>
    <button id="btn-t3split-{uid}" onclick="setT3Combine_{uid}(false)"
      style="padding:5px 14px;border-radius:20px;border:2px solid #1B4F8A;cursor:pointer;font-size:11px;
             font-weight:600;background:#fff;color:#1B4F8A;">Split</button>
  </div>
  <div style="position:relative;display:inline-block;
              background:linear-gradient(135deg,#e8f0f7 0%,#f0f5f9 100%);
              padding:20px 28px 28px;border-radius:12px;
              box-shadow:0 2px 14px rgba(0,0,0,0.09);overflow-x:auto;">
    <canvas id="canvas-{uid}"></canvas>
    <div id="tip-{uid}" style="display:none;position:fixed;pointer-events:none;
      background:rgba(10,30,55,0.92);color:#fff;padding:8px 12px;border-radius:6px;
      font-size:12px;box-shadow:0 3px 12px rgba(0,0,0,0.3);line-height:1.6;
      z-index:9999;max-width:260px;"></div>
  </div>
</div>

<script>
setTimeout(function(){{

const DATA = {payload};
let currentMode = 'detail';
let combineT2 = true;
let combineT3 = true;

const TOP=72,LPAD=16,FULL_H=1100,BOX_GAP=3,MIN_BOX=10,LEAF_GAP=2;
const ANC_W=110,SC_W=90,T1_W=175,T1L_W=78,T2_W=155,T2L_W=78,T3_W=145,T3L_W=78,GAP=82;
const ANC_X=LPAD,SC_X=ANC_X+ANC_W+GAP,T1_X=SC_X+SC_W+GAP,T1L_X=T1_X+T1_W+GAP;
const T2_X=T1L_X+T1L_W+GAP,T2L_X=T2_X+T2_W+GAP,T3_X=T2L_X+T2L_W+GAP,T3L_X=T3_X+T3_W+GAP;
function totalW(){{ return T3L_X+T3L_W+LPAD+10; }}
const TOTAL_H=TOP+FULL_H+40;

const S_COLORS=['#3A86C8','#2BAE96','#1B4F8A'];
const T1_COLORS=['#27AE8F','#2E86C1','#1D8A6E','#2471A3','#1A7A60','#1F6FA3','#17725A','#1A5F8C','#136150','#174F78','#8B7355'];
const T2_COLORS=['#5B9BD5','#70C1A8','#A0C4E8','#8FD8C4','#C2DFF2','#B0E8D8','#D8EEF8','#D0F0E8','#7BB8E0','#95D4BE','#E8C99A','#D4B07A','#C09060','#A87040','#9A6535','#8B5A2A'];
const T3_COLORS=['#F4A460','#DEB887','#CD853F','#D2691E','#8B4513','#A0522D','#BC8F5F','#E8A87C','#F5CBA7','#FAD7A0','#F0B27A','#E59866','#CA6F1E','#B7950B','#9A7D0A','#7D6608','#6E2F1A','#512E5F','#6C3483','#7D3C98'];

const C=document.getElementById('canvas-{uid}');
const DPR=window.devicePixelRatio||1;
let ctx;
function resizeCanvas(){{
  const W=totalW();
  C.width=W*DPR; C.height=TOTAL_H*DPR;
  C.style.width=W+'px'; C.style.height=TOTAL_H+'px';
  ctx=C.getContext('2d'); ctx.scale(DPR,DPR);
}}

function hexToRgb(hex){{ return [parseInt(hex.slice(1,3),16),parseInt(hex.slice(3,5),16),parseInt(hex.slice(5,7),16)]; }}
function rgbaStr(hex,a){{ const [r,g,b]=hexToRgb(hex); return `rgba(${{r}},${{g}},${{b}},${{a}})`; }}
function rRect(x,y,w,h,r){{
  ctx.beginPath();
  if(ctx.roundRect) ctx.roundRect(x,y,w,h,r);
  else {{ ctx.moveTo(x+r,y);ctx.lineTo(x+w-r,y);ctx.arcTo(x+w,y,x+w,y+r,r);ctx.lineTo(x+w,y+h-r);ctx.arcTo(x+w,y+h,x+w-r,y+h,r);ctx.lineTo(x+r,y+h);ctx.arcTo(x,y+h,x,y+h-r,r);ctx.lineTo(x,y+r);ctx.arcTo(x,y,x+r,y,r);ctx.closePath(); }}
}}
function proportionalHeights(pcts,totalH,gap){{
  const n=pcts.length,tot=pcts.reduce((a,b)=>a+b,0)||1;
  const available=totalH-gap*(n-1);
  let heights=pcts.map(p=>Math.max(MIN_BOX,(p/tot)*available));
  let diff=heights.reduce((a,b)=>a+b,0)-available;
  if(Math.abs(diff)>0.5){{
    const adj=heights.map((h,i)=>{{return{{i,h}}}}).filter(o=>o.h>MIN_BOX).sort((a,b)=>b.h-a.h);
    for(const o of adj){{const cut=Math.min(Math.abs(diff),heights[o.i]-MIN_BOX);heights[o.i]-=Math.sign(diff)*cut;diff-=Math.sign(diff)*cut;if(Math.abs(diff)<0.5)break;}}
  }}
  return heights;
}}
function drawBox(x,y,w,h,color,label,pct,val,fs){{
  fs=fs||9; const [r,g,b]=hexToRgb(color);
  const grad=ctx.createLinearGradient(x,y,x,y+h);
  grad.addColorStop(0,`rgba(${{Math.min(r+30,255)}},${{Math.min(g+30,255)}},${{Math.min(b+30,255)}},1)`);
  grad.addColorStop(1,color);
  ctx.fillStyle=grad; rRect(x,y,w,h,4); ctx.fill();
  ctx.strokeStyle=`rgba(${{Math.max(r-20,0)}},${{Math.max(g-20,0)}},${{Math.max(b-20,0)}},0.3)`;
  ctx.lineWidth=0.7; rRect(x,y,w,h,4); ctx.stroke();
  if(h<8) return;
  ctx.textBaseline='middle'; ctx.textAlign='left';
  const px=x+6,maxW=w-10;
  if(h>=22){{
    ctx.font=`600 ${{fs}}px 'Segoe UI',Arial,sans-serif`;
    ctx.fillStyle='rgba(255,255,255,0.95)';
    let txt=label;
    while(ctx.measureText(txt).width>maxW&&txt.length>3) txt=txt.slice(0,-1);
    if(txt!==label) txt=txt.slice(0,-1)+'…';
    ctx.fillText(txt,px,y+h*0.35);
    ctx.font=`${{fs-1}}px 'Segoe UI',Arial,sans-serif`;
    ctx.fillStyle='rgba(255,255,255,0.70)';
    ctx.fillText(pct+'%',px,y+h*0.68);
  }} else if(h>=14){{
    ctx.font=`${{fs-1}}px 'Segoe UI',Arial,sans-serif`;
    ctx.fillStyle='rgba(255,255,255,0.88)';
    ctx.fillText(pct+'%',px,y+h/2);
  }}
}}
function drawRibbon(x1,y1t,y1b,x2,y2t,y2b,color,alpha){{
  const mx=x1+(x2-x1)*0.52;
  ctx.beginPath(); ctx.moveTo(x1,y1t); ctx.bezierCurveTo(mx,y1t,mx,y2t,x2,y2t);
  ctx.lineTo(x2,y2b); ctx.bezierCurveTo(mx,y2b,mx,y1b,x1,y1b);
  ctx.closePath(); ctx.fillStyle=rgbaStr(color,alpha); ctx.fill();
}}
function drawHeader(x,w,l1,l2){{
  ctx.textAlign='center'; ctx.textBaseline='middle'; ctx.fillStyle='#0D2F4F';
  ctx.font="700 10px 'Segoe UI',Arial,sans-serif";
  ctx.fillText(l1,x+w/2,TOP-(l2?24:14));
  if(l2){{ ctx.font="400 8px 'Segoe UI',Arial,sans-serif"; ctx.fillStyle='#4A6A8A'; ctx.fillText(l2,x+w/2,TOP-11); }}
  ctx.strokeStyle='rgba(26,63,100,0.15)'; ctx.lineWidth=1;
  ctx.beginPath(); ctx.moveTo(x+4,TOP-3); ctx.lineTo(x+w-4,TOP-3); ctx.stroke();
}}

const tip=document.getElementById('tip-{uid}');
let hitBoxes=[];
function regHit(x,y,w,h,label,pct,val){{
  hitBoxes.push({{x,y,w,h,tip:`<b>${{label}}</b><br>${{pct}}% of total<br>${{val.toLocaleString()}} Mg CO\u2082e / M USD`}});
}}
C.addEventListener('mousemove',e=>{{
  const cr=C.getBoundingClientRect();
  const mx=(e.clientX-cr.left)*(totalW()/cr.width);
  const my=(e.clientY-cr.top)*(TOTAL_H/cr.height);
  const hit=hitBoxes.find(b=>mx>=b.x&&mx<=b.x+b.w&&my>=b.y&&my<=b.y+b.h);
  if(hit){{ tip.innerHTML=hit.tip; tip.style.display='block'; tip.style.left=(e.clientX+14)+'px'; tip.style.top=(e.clientY-10)+'px'; }}
  else tip.style.display='none';
}});
C.addEventListener('mouseleave',()=>tip.style.display='none');

function styleBtnPair(aId,bId){{
  [aId,bId].forEach((id,i)=>{{
    const el=document.getElementById(id);
    el.style.background=i===0?'#1B4F8A':'#fff';
    el.style.color=i===0?'#fff':'#1B4F8A';
    el.style.border=i===0?'none':'2px solid #1B4F8A';
  }});
}}
window.setMode_{uid}=function(mode){{
  currentMode=mode;
  styleBtnPair(mode==='detail'?'btn-detail-{uid}':'btn-summed-{uid}',mode==='detail'?'btn-summed-{uid}':'btn-detail-{uid}');
  render();
}};
window.setT2Combine_{uid}=function(c){{
  combineT2=c;
  styleBtnPair(c?'btn-t2combine-{uid}':'btn-t2split-{uid}',c?'btn-t2split-{uid}':'btn-t2combine-{uid}');
  render();
}};
window.setT3Combine_{uid}=function(c){{
  combineT3=c;
  styleBtnPair(c?'btn-t3combine-{uid}':'btn-t3split-{uid}',c?'btn-t3split-{uid}':'btn-t3combine-{uid}');
  render();
}};

function applyTxCombine(groups,itemsKey,combine){{
  if(!combine) return groups;
  const merged={{}};
  // Also track which t1names contribute to each merged label (for T3 key lookup)
  const mergedT1Names={{}};
  groups.forEach(g=>g[itemsKey].forEach(r=>{{
    if(!merged[r.label]) {{ merged[r.label]={{label:r.label,val:0,pct:0,s1_val:0,s1:0,s2_val:0,s2:0,s3_val:0,s3:0}}; mergedT1Names[r.label]=[]; }}
    const m=merged[r.label];
    m.val+=r.val;m.pct+=r.pct;m.s1_val+=r.s1_val;m.s1+=r.s1;m.s2_val+=r.s2_val;m.s2+=r.s2;m.s3_val+=r.s3_val;m.s3+=r.s3;
    // Store t1name for this group so T3 can look up all matching keys
    if(g.t1name&&!mergedT1Names[r.label].includes(g.t1name)) mergedT1Names[r.label].push(g.t1name);
  }}));
  const allItems=Object.values(merged).sort((a,b)=>b.val-a.val);
  // Attach t1names array to each merged item
  allItems.forEach(item=>{{ item.t1names=mergedT1Names[item.label]||[]; }});
  const fl=groups[0].sourceLeaf,ll=groups[groups.length-1].sourceLeaf;
  const combinedLeaf={{y:fl.y,h:(ll.y+ll.h)-fl.y,color:fl.color}};
  const result={{sourceLeaf:combinedLeaf,t1name:'Combined',t2name:'Combined'}};
  result[itemsKey]=allItems;
  return [result];
}}

function render(){{
  hitBoxes=[];
  ctx.clearRect(0,0,totalW(),TOTAL_H);

  drawHeader(ANC_X,ANC_W,'BASELINE','EMISSIONS');
  drawHeader(SC_X,SC_W,'SCOPE',null);
  drawHeader(T1_X,T1_W,'TIER 1','Input Sector');
  drawHeader(T1L_X,T1L_W,'TIER 1',currentMode==='detail'?'SCOPE (each)':'SCOPE (sum)');
  drawHeader(T2_X,T2_W,'TIER 2',combineT2?'Input Sector (combined)':'Input Sector');
  drawHeader(T2L_X,T2L_W,'TIER 2',currentMode==='detail'?'SCOPE (each)':'SCOPE (sum)');
  drawHeader(T3_X,T3_W,'TIER 3',combineT3?'Input Sector (combined)':'Input Sector');
  drawHeader(T3L_X,T3L_W,'TIER 3',currentMode==='detail'?'SCOPE (each)':'SCOPE (sum)');

  // Anchor chevron
  const chev=16;
  const aGrad=ctx.createLinearGradient(ANC_X,TOP,ANC_X,TOP+FULL_H);
  aGrad.addColorStop(0,'#1A4A72'); aGrad.addColorStop(1,'#0D2F4F');
  ctx.fillStyle=aGrad;
  ctx.beginPath(); ctx.moveTo(ANC_X,TOP); ctx.lineTo(ANC_X+ANC_W-chev,TOP);
  ctx.lineTo(ANC_X+ANC_W,TOP+FULL_H/2); ctx.lineTo(ANC_X+ANC_W-chev,TOP+FULL_H);
  ctx.lineTo(ANC_X,TOP+FULL_H); ctx.closePath(); ctx.fill();
  const ancCX=ANC_X+(ANC_W-chev)/2;
  ctx.textAlign='center'; ctx.textBaseline='middle';
  ctx.font="700 11px 'Segoe UI',Arial,sans-serif"; ctx.fillStyle='#fff';
  const words=DATA.sector.split(' '); let line='',lines=[];
  words.forEach(w=>{{if((line+' '+w).trim().length*6.5>ANC_W-20&&line){{lines.push(line);line=w;}}else line=(line+' '+w).trim();}});
  lines.push(line);
  lines.forEach((l,i)=>ctx.fillText(l,ancCX,TOP+FULL_H*0.38+(i-lines.length/2)*13));
  ctx.font="400 8px 'Segoe UI',Arial,sans-serif"; ctx.fillStyle='rgba(255,255,255,0.65)';
  ctx.fillText(DATA.total_ei.toLocaleString()+' Mg CO\u2082e/M$',ancCX,TOP+FULL_H*0.55);
  regHit(ANC_X,TOP,ANC_W,FULL_H,DATA.sector,100,DATA.total_ei);

  // Scope
  const scPcts=DATA.scopes.map(s=>s.pct);
  const scHeights=proportionalHeights(scPcts,FULL_H,BOX_GAP);
  let scY_arr=[],sy=TOP; scHeights.forEach(h=>{{scY_arr.push(sy);sy+=h+BOX_GAP;}});
  let ancCursor=TOP;
  DATA.scopes.forEach((sc,si)=>{{
    const share=sc.pct/scPcts.reduce((a,b)=>a+b,0);
    drawRibbon(ANC_X+ANC_W-chev,ancCursor,ancCursor+share*FULL_H,SC_X,scY_arr[si],scY_arr[si]+scHeights[si],sc.color,0.22);
    ancCursor+=share*FULL_H;
  }});
  const scopeBoxes=[];
  DATA.scopes.forEach((sc,si)=>{{
    drawBox(SC_X,scY_arr[si],SC_W,scHeights[si],sc.color,sc.label,sc.pct,sc.val,10);
    regHit(SC_X,scY_arr[si],SC_W,scHeights[si],sc.label,sc.pct,sc.val);
    scopeBoxes.push({{y:scY_arr[si],h:scHeights[si],color:sc.color,pct:sc.pct,val:sc.val}});
  }});

  // Tier 1
  const t1Pcts=DATA.tier1.map(t=>t.pct);
  const t1Heights=proportionalHeights(t1Pcts,FULL_H,BOX_GAP);
  const s3box=scopeBoxes[2]; let t1Boxes=[],t1Y=TOP;
  let s3cursor=s3box.y; const s3tot=s3box.pct||1;
  DATA.tier1.forEach((t,i)=>{{
    const h=t1Heights[i],col=T1_COLORS[i%T1_COLORS.length];
    const sliceH=(t.pct/s3tot)*s3box.h;
    drawRibbon(SC_X+SC_W,s3cursor,s3cursor+sliceH,T1_X,t1Y,t1Y+h,col,0.20);
    s3cursor+=sliceH; t1Y+=h+BOX_GAP;
  }});
  t1Y=TOP;
  DATA.tier1.forEach((t,i)=>{{
    const h=t1Heights[i],col=T1_COLORS[i%T1_COLORS.length];
    t1Boxes.push({{y:t1Y,h,color:col,pct:t.pct,val:t.val,label:t.label}});
    drawBox(T1_X,t1Y,T1_W,h,col,t.label,t.pct,t.val,9);
    regHit(T1_X,t1Y,T1_W,h,'Tier 1: '+t.label,t.pct,t.val);
    t1Y+=h+BOX_GAP;
  }});

  // Tier 1 scope leaves
  let t1LeafBoxes=[];
  if(currentMode==='detail'){{
    const allLPcts=[]; DATA.tier1.forEach(t=>allLPcts.push(t.s1,t.s2,t.s3));
    const N=allLPcts.length; const lAvail=FULL_H-LEAF_GAP*(N-1);
    const lTot=allLPcts.reduce((a,b)=>a+b,0)||1;
    const MIN_L=Math.max(4,lAvail/N*0.20);
    let allLH=allLPcts.map(p=>Math.max(MIN_L,(p/lTot)*lAvail));
    const lsum=allLH.reduce((a,b)=>a+b,0); allLH=allLH.map(h=>h/lsum*lAvail);
    let fi=0,leafY=TOP;
    DATA.tier1.forEach((t,i)=>{{
      const tb=t1Boxes[i],lPcts=[t.s1,t.s2,t.s3],lVals=[t.s1_val,t.s2_val,t.s3_val];
      const lhs=[allLH[fi],allLH[fi+1],allLH[fi+2]];
      const ptot=lPcts.reduce((a,b)=>a+b,0)||1; let t1cur=tb.y;
      lhs.forEach((lh,si)=>{{
        const sliceH=(lPcts[si]/ptot)*tb.h;
        drawRibbon(T1_X+T1_W,t1cur,t1cur+sliceH,T1L_X,leafY,leafY+lh,S_COLORS[si],0.20);
        t1cur+=sliceH;
        t1LeafBoxes.push({{y:leafY,h:lh,color:S_COLORS[si],pct:lPcts[si],val:lVals[si],t1idx:i,si}});
        leafY+=lh+LEAF_GAP;
      }});
      fi+=3;
    }});
    t1LeafBoxes.forEach(b=>{{
      drawBox(T1L_X,b.y,T1L_W,b.h,b.color,['Scope 1','Scope 2','Scope 3'][b.si],b.pct,b.val,8);
      regHit(T1L_X,b.y,T1L_W,b.h,DATA.tier1[b.t1idx].label+' \u00b7 '+['Scope 1','Scope 2','Scope 3'][b.si],b.pct,b.val);
    }});
  }} else {{
    const smP=[DATA.tier1.reduce((a,t)=>a+t.s1,0),DATA.tier1.reduce((a,t)=>a+t.s2,0),DATA.tier1.reduce((a,t)=>a+t.s3,0)];
    const smV=[DATA.tier1.reduce((a,t)=>a+t.s1_val,0),DATA.tier1.reduce((a,t)=>a+t.s2_val,0),DATA.tier1.reduce((a,t)=>a+t.s3_val,0)];
    const smH=proportionalHeights(smP,FULL_H,BOX_GAP);
    DATA.tier1.forEach((t,i)=>{{
      const tb=t1Boxes[i],lP=[t.s1,t.s2,t.s3],ptot=lP.reduce((a,b)=>a+b,0)||1;
      let t1cur=tb.y,bucketY=TOP;
      smH.forEach((bh,si)=>{{
        const sliceH=(lP[si]/ptot)*tb.h;
        drawRibbon(T1_X+T1_W,t1cur,t1cur+sliceH,T1L_X,bucketY,bucketY+bh,S_COLORS[si],0.12);
        t1cur+=sliceH; bucketY+=bh+BOX_GAP;
      }});
    }});
    let smY=TOP;
    smH.forEach((h,si)=>{{
      const lbl=['Scope 1','Scope 2','Scope 3'][si];
      drawBox(T1L_X,smY,T1L_W,h,S_COLORS[si],lbl,smP[si],smV[si],9);
      regHit(T1L_X,smY,T1L_W,h,'Tier 1 '+lbl+' (all)',smP[si],smV[si]);
      t1LeafBoxes.push({{y:smY,h,color:S_COLORS[si],pct:smP[si],val:smV[si],t1idx:-1,si}});
      smY+=h+BOX_GAP;
    }});
  }}

  // Tier 2 sectors
  const t2RawGroups=[];
  if(currentMode==='detail'){{
    t1LeafBoxes.filter(b=>b.si===2).forEach((leaf,i)=>{{
      const t1name=DATA.tier1[leaf.t1idx].label;
      t2RawGroups.push({{t1idx:leaf.t1idx,t2items:DATA.tier2[t1name]||[],sourceLeaf:leaf,t1name}});
    }});
  }} else {{
    const s3Leaf=t1LeafBoxes.find(b=>b.si===2);
    const s3Total=DATA.tier1.reduce((a,x)=>a+x.s3,0)||1;
    let cursor=s3Leaf.y;
    DATA.tier1.forEach((t,i)=>{{
      const share=t.s3/s3Total;
      t2RawGroups.push({{t1idx:i,t2items:DATA.tier2[t.label]||[],sourceLeaf:{{y:cursor,h:s3Leaf.h*share,color:s3Leaf.color}},t1name:t.label}});
      cursor+=s3Leaf.h*share;
    }});
  }}
  const t2Groups=applyTxCombine(t2RawGroups,'t2items',combineT2);

  const allT2P=[]; t2Groups.forEach(g=>g.t2items.forEach(r=>allT2P.push(r.pct)));
  const totT2P=allT2P.reduce((a,b)=>a+b,0)||1;
  const t2AvH=FULL_H-LEAF_GAP*(allT2P.length-1);
  const MIN_T2=Math.max(4,t2AvH/Math.max(allT2P.length,1)*0.15);
  let allT2H=allT2P.map(p=>Math.max(MIN_T2,(p/totT2P)*t2AvH));
  allT2H=allT2H.map(h=>h/allT2H.reduce((a,b)=>a+b,0)*t2AvH);

  let t2Y=TOP,ft2=0; const t2BoxList=[];
  t2Groups.forEach((g,gi)=>{{
    const src=g.sourceLeaf; let srcC=src.y;
    const srcTot=g.t2items.reduce((a,r)=>a+r.pct,0)||1;
    g.t2items.forEach((r,ri)=>{{
      const h=allT2H[ft2],col=T2_COLORS[(gi*4+ri)%T2_COLORS.length];
      const sliceH=(r.pct/srcTot)*src.h;
      drawRibbon(T1L_X+T1L_W,srcC,srcC+sliceH,T2_X,t2Y,t2Y+h,col,0.20);
      srcC+=sliceH;
      const t1names=r.t1names&&r.t1names.length?r.t1names:[g.t1name];
      t2BoxList.push({{y:t2Y,h,color:col,t2:r,gi,ri,t1name:g.t1name,t1names}});
      t2Y+=h+LEAF_GAP; ft2++;
    }});
  }});
  t2BoxList.forEach(b=>{{
    drawBox(T2_X,b.y,T2_W,b.h,b.color,b.t2.label,b.t2.pct,b.t2.val,8);
    regHit(T2_X,b.y,T2_W,b.h,'Tier 2: '+b.t2.label,b.t2.pct,b.t2.val);
  }});

  // Tier 2 scope leaves
  let t2LeafBoxes=[];
  if(currentMode==='detail'){{
    const allT2LP=[]; t2BoxList.forEach(b=>allT2LP.push(b.t2.s1,b.t2.s2,b.t2.s3));
    const t2LAv=FULL_H-LEAF_GAP*(allT2LP.length-1);
    const t2LTot=allT2LP.reduce((a,b)=>a+b,0)||1;
    const MIN_T2L=Math.max(3,t2LAv/Math.max(allT2LP.length,1)*0.10);
    let allT2LH=allT2LP.map(p=>Math.max(MIN_T2L,(p/t2LTot)*t2LAv));
    allT2LH=allT2LH.map(h=>h/allT2LH.reduce((a,b)=>a+b,0)*t2LAv);
    let t2LY=TOP,fL=0;
    t2BoxList.forEach((b,bi)=>{{
      const lP=[b.t2.s1,b.t2.s2,b.t2.s3],lV=[b.t2.s1_val,b.t2.s2_val,b.t2.s3_val];
      const ptot=lP.reduce((a,x)=>a+x,0)||1; let t2cur=b.y;
      lP.forEach((p,si)=>{{
        const lh=allT2LH[fL],sliceH=(p/ptot)*b.h;
        drawRibbon(T2_X+T2_W,t2cur,t2cur+sliceH,T2L_X,t2LY,t2LY+lh,S_COLORS[si],0.20);
        drawBox(T2L_X,t2LY,T2L_W,lh,S_COLORS[si],['S1','S2','S3'][si],p,lV[si],7);
        regHit(T2L_X,t2LY,T2L_W,lh,'Tier 2: '+b.t2.label+' \u00b7 '+['Scope 1','Scope 2','Scope 3'][si],p,lV[si]);
        t2LeafBoxes.push({{y:t2LY,h:lh,color:S_COLORS[si],pct:p,val:lV[si],bi,si,t1name:b.t1name,t1names:b.t1names,t2label:b.t2.label}});
        t2cur+=sliceH; t2LY+=lh+LEAF_GAP; fL++;
      }});
    }});
  }} else {{
    const sm2P=[t2BoxList.reduce((a,b)=>a+b.t2.s1,0),t2BoxList.reduce((a,b)=>a+b.t2.s2,0),t2BoxList.reduce((a,b)=>a+b.t2.s3,0)];
    const sm2V=[t2BoxList.reduce((a,b)=>a+b.t2.s1_val,0),t2BoxList.reduce((a,b)=>a+b.t2.s2_val,0),t2BoxList.reduce((a,b)=>a+b.t2.s3_val,0)];
    const sm2H=proportionalHeights(sm2P,FULL_H,BOX_GAP);
    let bTops=[]; let bC=TOP; sm2H.forEach(h=>{{bTops.push(bC);bC+=h+BOX_GAP;}});
    t2BoxList.forEach(b=>{{
      const lP=[b.t2.s1,b.t2.s2,b.t2.s3],ptot=lP.reduce((a,x)=>a+x,0)||1; let t2cur=b.y;
      lP.forEach((p,si)=>{{const sliceH=(p/ptot)*b.h;drawRibbon(T2_X+T2_W,t2cur,t2cur+sliceH,T2L_X,bTops[si],bTops[si]+sm2H[si],S_COLORS[si],0.12);t2cur+=sliceH;}});
    }});
    let bY2=TOP;
    sm2H.forEach((h,si)=>{{
      const lbl=['Scope 1','Scope 2','Scope 3'][si];
      drawBox(T2L_X,bY2,T2L_W,h,S_COLORS[si],lbl,sm2P[si],sm2V[si],9);
      regHit(T2L_X,bY2,T2L_W,h,'Tier 2 '+lbl+' (all)',sm2P[si],sm2V[si]);
      bY2+=h+BOX_GAP;
    }});
    // For Tier 3 source: carve S3 bucket proportionally per t2 box
    const s3TotT2=t2BoxList.reduce((a,b)=>a+b.t2.s3,0)||1;
    const s3BH=sm2H[2],s3BY=bTops[2]; let c3=s3BY;
    t2BoxList.forEach((b,bi)=>{{
      const share=b.t2.s3/s3TotT2,lh=s3BH*share;
      t2LeafBoxes.push({{y:c3,h:lh,color:S_COLORS[2],pct:b.t2.s3,val:b.t2.s3_val,bi,si:2,t1name:b.t1name,t1names:b.t1names,t2label:b.t2.label}});
      c3+=lh;
    }});
  }}

  // Tier 3 sectors
  // Helper: collect t3 items across all t1 variants for a given t2 label
  function getT3Items(leaf){{
    const t2label=leaf.t2label;
    const t1names=leaf.t1names&&leaf.t1names.length?leaf.t1names:[leaf.t1name];
    // Merge all matching keys (t1name|||t2label) into one de-duped item list
    const merged={{}};
    t1names.forEach(t1n=>{{
      const key=t1n+'|||'+t2label;
      (DATA.tier3[key]||[]).forEach(r=>{{
        if(!merged[r.label]) merged[r.label]={{...r}};
        else {{
          const m=merged[r.label];
          m.val+=r.val;m.pct+=r.pct;
          m.s1_val+=r.s1_val;m.s1+=r.s1;
          m.s2_val+=r.s2_val;m.s2+=r.s2;
          m.s3_val+=r.s3_val;m.s3+=r.s3;
        }}
      }});
    }});
    return Object.values(merged).sort((a,b)=>b.val-a.val);
  }}
  const s3T2Leaves=(currentMode==='detail')?t2LeafBoxes.filter(b=>b.si===2):t2LeafBoxes;
  const t3RawGroups=[];
  s3T2Leaves.forEach((leaf,i)=>{{
    const t3items=getT3Items(leaf);
    t3RawGroups.push({{t2idx:leaf.bi,t3items,sourceLeaf:leaf,t1name:leaf.t1name,t2name:leaf.t2label}});
  }});
  const t3Groups=applyTxCombine(t3RawGroups,'t3items',combineT3);

  const allT3P=[]; t3Groups.forEach(g=>g.t3items.forEach(r=>allT3P.push(r.pct)));
  const totT3P=allT3P.reduce((a,b)=>a+b,0)||1;
  const t3AvH=FULL_H-LEAF_GAP*(allT3P.length-1);
  const MIN_T3=Math.max(4,t3AvH/Math.max(allT3P.length,1)*0.15);
  let allT3H=allT3P.map(p=>Math.max(MIN_T3,(p/totT3P)*t3AvH));
  allT3H=allT3H.map(h=>h/allT3H.reduce((a,b)=>a+b,0)*t3AvH);

  let t3Y=TOP,ft3=0; const t3BoxList=[];
  t3Groups.forEach((g,gi)=>{{
    const src=g.sourceLeaf; let srcC=src.y;
    const srcTot=g.t3items.reduce((a,r)=>a+r.pct,0)||1;
    g.t3items.forEach((r,ri)=>{{
      const h=allT3H[ft3],col=T3_COLORS[(gi*4+ri)%T3_COLORS.length];
      const sliceH=(r.pct/srcTot)*src.h;
      drawRibbon(T2L_X+T2L_W,srcC,srcC+sliceH,T3_X,t3Y,t3Y+h,col,0.20);
      srcC+=sliceH;
      t3BoxList.push({{y:t3Y,h,color:col,t3:r,gi,ri}});
      t3Y+=h+LEAF_GAP; ft3++;
    }});
  }});
  t3BoxList.forEach(b=>{{
    drawBox(T3_X,b.y,T3_W,b.h,b.color,b.t3.label,b.t3.pct,b.t3.val,8);
    regHit(T3_X,b.y,T3_W,b.h,'Tier 3: '+b.t3.label,b.t3.pct,b.t3.val);
  }});

  // Tier 3 scope leaves
  if(currentMode==='detail'){{
    const allT3LP=[]; t3BoxList.forEach(b=>allT3LP.push(b.t3.s1,b.t3.s2,b.t3.s3));
    const t3LAv=FULL_H-LEAF_GAP*(allT3LP.length-1);
    const t3LTot=allT3LP.reduce((a,b)=>a+b,0)||1;
    const MIN_T3L=Math.max(3,t3LAv/Math.max(allT3LP.length,1)*0.08);
    let allT3LH=allT3LP.map(p=>Math.max(MIN_T3L,(p/t3LTot)*t3LAv));
    allT3LH=allT3LH.map(h=>h/allT3LH.reduce((a,b)=>a+b,0)*t3LAv);
    let t3LY=TOP,fL3=0;
    t3BoxList.forEach(b=>{{
      const lP=[b.t3.s1,b.t3.s2,b.t3.s3],lV=[b.t3.s1_val,b.t3.s2_val,b.t3.s3_val];
      const ptot=lP.reduce((a,x)=>a+x,0)||1; let t3cur=b.y;
      lP.forEach((p,si)=>{{
        const lh=allT3LH[fL3],sliceH=(p/ptot)*b.h;
        drawRibbon(T3_X+T3_W,t3cur,t3cur+sliceH,T3L_X,t3LY,t3LY+lh,S_COLORS[si],0.20);
        drawBox(T3L_X,t3LY,T3L_W,lh,S_COLORS[si],['S1','S2','S3'][si],p,lV[si],7);
        regHit(T3L_X,t3LY,T3L_W,lh,'Tier 3: '+b.t3.label+' \u00b7 '+['Scope 1','Scope 2','Scope 3'][si],p,lV[si]);
        t3cur+=sliceH; t3LY+=lh+LEAF_GAP; fL3++;
      }});
    }});
  }} else {{
    const sm3P=[t3BoxList.reduce((a,b)=>a+b.t3.s1,0),t3BoxList.reduce((a,b)=>a+b.t3.s2,0),t3BoxList.reduce((a,b)=>a+b.t3.s3,0)];
    const sm3V=[t3BoxList.reduce((a,b)=>a+b.t3.s1_val,0),t3BoxList.reduce((a,b)=>a+b.t3.s2_val,0),t3BoxList.reduce((a,b)=>a+b.t3.s3_val,0)];
    const sm3H=proportionalHeights(sm3P,FULL_H,BOX_GAP);
    let bT3=[]; let bC3=TOP; sm3H.forEach(h=>{{bT3.push(bC3);bC3+=h+BOX_GAP;}});
    t3BoxList.forEach(b=>{{
      const lP=[b.t3.s1,b.t3.s2,b.t3.s3],ptot=lP.reduce((a,x)=>a+x,0)||1; let t3cur=b.y;
      lP.forEach((p,si)=>{{const sliceH=(p/ptot)*b.h;drawRibbon(T3_X+T3_W,t3cur,t3cur+sliceH,T3L_X,bT3[si],bT3[si]+sm3H[si],S_COLORS[si],0.12);t3cur+=sliceH;}});
    }});
    let bY3=TOP;
    sm3H.forEach((h,si)=>{{
      const lbl=['Scope 1','Scope 2','Scope 3'][si];
      drawBox(T3L_X,bY3,T3L_W,h,S_COLORS[si],lbl,sm3P[si],sm3V[si],9);
      regHit(T3L_X,bY3,T3L_W,h,'Tier 3 '+lbl+' (all)',sm3P[si],sm3V[si]);
      bY3+=h+BOX_GAP;
    }});
  }}

  ctx.font="400 8px 'Segoe UI',Arial,sans-serif";
  ctx.fillStyle='rgba(80,110,140,0.60)';
  ctx.textAlign='left'; ctx.textBaseline='top';
  ctx.fillText('Emissions Intensity (Mg CO\u2082e per Million USD) \u00b7 '+DATA.sector,ANC_X,TOP+FULL_H+10);
}}

resizeCanvas();
render();
}}, 150);
</script>
"""

display(HTML(html))
